# 📓 Notebook 14 — Time Series and Forecasting Basics

> **Module:** Applied Data Science · **Estimated time:** 60–75 min · **Difficulty:** Intermediate

Most of the data a business runs on is **time-shaped**: revenue per day, tickets per hour, signups per week, clicks per minute. Yet up to now the course has treated rows as exchangeable. This notebook fixes that.

You will learn the small set of pandas idioms (resampling, rolling windows, date arithmetic) that make time-series work feel natural, then build a real **3-month forecast** of the AI support-bot's automation rate using both classical methods (naive baseline, Holt-Winters) and a touch of ARIMA — and learn to *evaluate* a forecast honestly.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Convert string dates to `datetime`, set a `DatetimeIndex`, and select date ranges.
2. **Resample** from daily → weekly → monthly with the right aggregation per column.
3. Compute **rolling means** and **expanding sums** for trend smoothing.
4. **Decompose** a series into trend + seasonality + residual.
5. Build three forecasts: a **naive baseline**, a **moving-average baseline**, and **Holt-Winters exponential smoothing**.
6. Use **walk-forward backtesting** to evaluate forecasts honestly.
7. Decide which baseline to beat before declaring a model "good".

## ✅ Prerequisites

Notebooks 1–8 (especially NB5/NB7 for pandas + numpy, NB8 for plotting).

## 1. The shape of a time series

```
   date           value
   ────────       ─────
   2024-01-01     0.52
   2024-01-02     0.54
   2024-01-03     0.55
   2024-01-04     0.51
   ...            ...
```

Two things make a time series different from a normal table:

1. **The index is time** — rows have a natural order; you can ask *"what's the value 7 days from now?"*
2. **Consecutive rows are correlated** — yesterday's automation rate is the strongest predictor of today's. This breaks every "rows are independent" assumption that classical statistics is built on.

The good news: pandas was *designed* for time series. Once a DataFrame has a `DatetimeIndex`, an entire set of time-aware operations becomes available.

## 2. Setup — generate two years of daily data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

RNG = np.random.default_rng(42)

# Two years of daily data for the Chat channel
n_days = 730
dates  = pd.date_range("2023-01-01", periods=n_days, freq="D")

# Build a realistic series: trend up + weekly seasonality + small noise + a holiday dip
t      = np.arange(n_days)
trend  = 0.50 + 0.0003 * t                            # slow upward drift
weekly = 0.05 * np.sin(2 * np.pi * t / 7)             # peak Wednesdays, dip Sundays
yearly = 0.03 * np.sin(2 * np.pi * t / 365.25 - 1.5)  # mild summer dip
noise  = RNG.normal(0, 0.015, n_days)
auto_rate = trend + weekly + yearly + noise

# Add a "Christmas dip" each year
for year in (2023, 2024):
    holiday_mask = (dates >= f"{year}-12-22") & (dates <= f"{year}-12-29")
    auto_rate[holiday_mask] -= 0.08

# Build a DataFrame with a DatetimeIndex
df = pd.DataFrame({"date": dates, "auto_rate": auto_rate.clip(0, 1)})
df = df.set_index("date")
print(df.head())
print(f"\nShape: {df.shape}")
print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")


> 💡 **`pd.date_range` and `set_index`** are the two functions you'll use every time you start with a CSV that has a date column. Together they convert a flat table into a *time-indexed* one — which unlocks everything else in this notebook.

## 3. Selecting by date

Once the index is a `DatetimeIndex`, you can slice it with human-readable strings.

In [ ]:
# One specific day
print("One day :", df.loc["2024-03-15"].iloc[0])

# A whole month
march = df.loc["2024-03"]
print(f"\nMarch 2024:  {len(march)} rows, mean = {march['auto_rate'].mean():.3f}")

# A range
q2 = df.loc["2024-04-01":"2024-06-30"]
print(f"Q2 2024  :  {len(q2)} rows, mean = {q2['auto_rate'].mean():.3f}")

# Last 30 days of data
print(f"\nLast 30 days mean: {df.tail(30)['auto_rate'].mean():.3f}")


**Quick wins from a `DatetimeIndex`:**

- `df.loc["2024-03"]` — partial-string matching: "all rows whose date starts with 2024-03".
- `df.loc["2024-04-01":"2024-06-30"]` — both endpoints **inclusive** (unlike normal Python slicing).
- `df.index.dayofweek`, `df.index.month`, `df.index.is_quarter_end` — vectorised date attributes.

## 4. Resampling — aggregate from daily to weekly / monthly

`.resample(rule).agg(...)` is the time-series sibling of `groupby`. The `rule` is a frequency code: `"D"` daily, `"W"` weekly, `"ME"` month-end, `"QE"` quarter-end, `"YE"` year-end.

In [ ]:
# Daily → weekly mean
weekly = df.resample("W").mean()
print(f"Weekly view: {len(weekly)} weeks")
print(weekly.head())

# Daily → monthly with several aggregations at once
monthly = df.resample("ME").agg(
    mean_auto = ("auto_rate", "mean"),
    min_auto  = ("auto_rate", "min"),
    max_auto  = ("auto_rate", "max"),
)
print(f"\nMonthly view: {len(monthly)} months")
print(monthly.head().round(3))


In [ ]:
# Visualise: daily raw vs weekly smoothed
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(df.index, df["auto_rate"], alpha=0.35, label="daily (noisy)", color="#7F7F7F")
ax.plot(weekly.index, weekly["auto_rate"], lw=2, color="#4C72B0", label="weekly mean")
ax.set_title("Daily automation rate vs weekly average")
ax.set_ylabel("Automation rate")
ax.legend()
plt.tight_layout(); plt.show()


> 🎯 **Resampling is your first weapon against noise.** Daily data is jittery; weekly or monthly aggregates expose the underlying signal. The right granularity depends on the question — daily for operations, weekly for trends, monthly for strategy.

## 5. Rolling windows — smoothing without losing rows

Resampling collapses many rows into one. **Rolling windows** keep every row but replace each value with an aggregate of the surrounding rows.

In [ ]:
# 7-day and 30-day rolling means
df["roll_7"]  = df["auto_rate"].rolling(window=7,  min_periods=1).mean()
df["roll_30"] = df["auto_rate"].rolling(window=30, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(df.index, df["auto_rate"], alpha=0.35, color="#7F7F7F", label="daily")
ax.plot(df.index, df["roll_7"],   lw=1.6, color="#55A467", label="7-day rolling mean")
ax.plot(df.index, df["roll_30"],  lw=2,   color="#C44E52", label="30-day rolling mean")
ax.set_title("Rolling means smooth the signal at different scales")
ax.set_ylabel("Automation rate")
ax.legend()
plt.tight_layout(); plt.show()


**Rolling vs resampling — the difference:**

- *Resampling* changes the time grid (one row per week instead of one per day).
- *Rolling* preserves the grid but each value is now a window aggregate.

> 💡 **`min_periods`** controls what happens at the edges. `min_periods=1` lets the window shrink at the start (so the first 6 days don't become `NaN`). Drop it for stricter behaviour.

## 6. Date arithmetic — `.shift` and `.diff`

Two operations that show up constantly:

- `s.shift(k)` — slide the series down by `k` rows (lag/lead).
- `s.diff(k)` — `s - s.shift(k)`. The change from `k` periods ago.

In [ ]:
# Day-over-day change
df["delta_1d"] = df["auto_rate"].diff(1)

# 7-day momentum (today vs the same day last week)
df["delta_7d"] = df["auto_rate"].diff(7)

print(df[["auto_rate", "delta_1d", "delta_7d"]].tail(10).round(3))

# Plot the 7-day delta — exposes the weekly seasonality
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df["delta_7d"], lw=0.9, color="#8172B2", alpha=0.7)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Seven-day change (auto_rate today − auto_rate 7 days ago)")
ax.set_ylabel("Δ automation rate")
plt.tight_layout(); plt.show()


> 🎯 **Why differencing matters.** A noisy trending series often has a *much* more stable difference. For forecasting, this is critical — most classical models assume the series is "stationary" (constant mean, constant variance), and `.diff()` is your first stationarity tool.

## 7. Decomposition — trend + seasonality + residual

Any time series can be (approximately) split into three pieces:

```
observed = trend + seasonality + residual
```

`statsmodels` does this in one call.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Decompose with weekly seasonality (period = 7)
decomp = seasonal_decompose(df["auto_rate"], model="additive", period=7,
                             extrapolate_trend="freq")

fig, axes = plt.subplots(4, 1, figsize=(11, 8), sharex=True)
df["auto_rate"].plot(ax=axes[0], color="#7F7F7F"); axes[0].set_title("Observed")
decomp.trend.plot(ax=axes[1], color="#4C72B0");    axes[1].set_title("Trend")
decomp.seasonal.plot(ax=axes[2], color="#55A467"); axes[2].set_title("Seasonality (weekly)")
decomp.resid.plot(ax=axes[3], color="#C44E52");    axes[3].set_title("Residual")
plt.tight_layout(); plt.show()


**Reading the four panels.**

- **Observed** — the raw series, jittery.
- **Trend** — the slow upward drift we baked into the data.
- **Seasonality** — the weekly cycle, repeated.
- **Residual** — everything else, ideally noise. If residuals show structure, your decomposition missed something (e.g., a second seasonality, an outlier, a regime change).

> 💡 **Decomposition before modelling.** A 5-minute decomposition plot tells you which patterns a model needs to capture — and which ones it can ignore. Worth the time on every new time series you meet.

## 8. Forecasting — three baselines you must beat

Before you reach for fancy models, you must beat the **trivial** baselines. Any model that doesn't beat a naive forecast isn't earning its complexity.

In [ ]:
# Use the FIRST 700 days for training, hold out the LAST 30 for testing
train = df.iloc[:700]["auto_rate"]
test  = df.iloc[700:]["auto_rate"]
h     = len(test)         # horizon = 30 days
print(f"Train: {len(train)} days   Test: {h} days")


### Baseline 1 — *Naive*: tomorrow looks like today

In [ ]:
# Repeat the last observed value for the whole horizon
last_value   = train.iloc[-1]
fc_naive     = pd.Series(last_value, index=test.index, name="naive")
print(f"Naive forecast: constant {last_value:.3f} for {h} days")


### Baseline 2 — *Seasonal naive*: this day-of-week looks like last week's

In [ ]:
# For each test date, predict the value from 7 days earlier
fc_snaive = test.copy() * 0.0   # same index, zeroed
for i, ts in enumerate(test.index):
    fc_snaive.iloc[i] = train.iloc[-7 + (i % 7)] if i < 7 else fc_snaive.iloc[i - 7]
fc_snaive.name = "snaive"

# Quick sanity check
print(fc_snaive.head(10).round(3))


### Baseline 3 — *Moving average*: this week's average extended

In [ ]:
# Last 30-day mean carried forward
fc_ma = pd.Series(train.tail(30).mean(), index=test.index, name="ma30")
print(f"30-day moving-average forecast: {fc_ma.iloc[0]:.3f}")


### Holt-Winters — a proper baseline that handles trend + seasonality

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Additive trend + weekly seasonal
model = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=7,
                              initialization_method="estimated")
fit   = model.fit()
fc_hw = fit.forecast(h)
fc_hw.name = "holt_winters"

# Inspect the fitted smoothing parameters
print(f"alpha (level)   = {fit.params['smoothing_level']:.3f}")
print(f"beta  (trend)   = {fit.params['smoothing_trend']:.3f}")
print(f"gamma (season)  = {fit.params['smoothing_seasonal']:.3f}")


## 9. Visualising the forecasts

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train.tail(60).index, train.tail(60), color="#7F7F7F", lw=1.5, label="train (last 60d)")
ax.plot(test.index, test,                       color="black", lw=2,   label="actual")
ax.plot(fc_naive.index,  fc_naive,  color="#4C72B0", ls="--", lw=2, label="naive")
ax.plot(fc_snaive.index, fc_snaive, color="#55A467", ls="--", lw=2, label="seasonal naive")
ax.plot(fc_ma.index,     fc_ma,     color="#8172B2", ls="--", lw=2, label="30-day MA")
ax.plot(fc_hw.index,     fc_hw,     color="#C44E52", lw=2.2,  label="Holt-Winters")
ax.axvline(test.index[0], color="orange", lw=1, ls=":")
ax.set_title("Forecasts for the 30-day hold-out period")
ax.set_ylabel("Automation rate")
ax.legend(ncol=2, fontsize=9)
plt.tight_layout(); plt.show()


## 10. Evaluating forecasts — MAE, RMSE, MAPE

Three metrics cover most cases:

- **MAE** (mean absolute error) — in target units. Easy to communicate.
- **RMSE** (root mean squared error) — like MAE but penalises big errors.
- **MAPE** (mean absolute percentage error) — unit-free; compares across series of different scales.

In [ ]:
def evaluate(y_true, y_pred, name):
    mae  = (y_true - y_pred).abs().mean()
    rmse = ((y_true - y_pred) ** 2).mean() ** 0.5
    mape = ((y_true - y_pred).abs() / y_true.abs()).mean() * 100
    return {"model": name, "MAE": mae, "RMSE": rmse, "MAPE %": mape}


rows = []
for fc, name in [(fc_naive, "naive"), (fc_snaive, "seasonal naive"),
                 (fc_ma, "30-day MA"), (fc_hw, "Holt-Winters")]:
    rows.append(evaluate(test, fc, name))

results = pd.DataFrame(rows).set_index("model").round(4)
print(results.sort_values("MAE"))


**How to read this table.**

- The model with the **lowest MAE / RMSE** is winning *on this hold-out*.
- A single hold-out is one data point. For trust, do **walk-forward backtesting** (next section).
- MAPE in single digits is good for most business forecasts. Above 20% the forecast is little better than a hand-wave.

> 🎯 **The "you must beat" line.** If your shiny new model can't beat *seasonal naive*, the model is not the problem — your data has very little structure to learn, or your features are weak. Walk away with the simple model.

## 11. Walk-forward backtesting — the honest evaluation

A single train/test split can be lucky. Walk-forward refits the model on a growing window and forecasts a small step at a time, mimicking how the forecast would actually be used in production.

In [ ]:
# Cheaper version: refit Holt-Winters every 30 days, forecast next 30
def walk_forward(series, model_fn, horizon=30, step=30, min_train=180):
    """Refit at each step; forecast `horizon` ahead; return a DataFrame of errors."""
    errs = []
    for start in range(min_train, len(series) - horizon, step):
        train = series.iloc[:start]
        test  = series.iloc[start:start + horizon]
        try:
            preds = model_fn(train, horizon)
            errs.append({
                "anchor_date": series.index[start].date(),
                "MAE":  (test - preds).abs().mean(),
                "RMSE": ((test - preds) ** 2).mean() ** 0.5,
            })
        except Exception as e:
            print(f"  fold @ {series.index[start].date()}: {type(e).__name__}: {e}")
    return pd.DataFrame(errs)


def hw_forecast(train, h):
    m = ExponentialSmoothing(train, trend="add", seasonal="add",
                              seasonal_periods=7,
                              initialization_method="estimated")
    return m.fit().forecast(h)


bt = walk_forward(df["auto_rate"], hw_forecast, horizon=30, step=60, min_train=200)
print(bt.round(4))
print(f"\nHolt-Winters — average over {len(bt)} folds: "
      f"MAE = {bt['MAE'].mean():.4f}, RMSE = {bt['RMSE'].mean():.4f}")


You now have a number that *generalises* — the average error across many different hold-out windows. That's the honest answer to "how well will this forecast do on next month?", not "how well did it do on the one hold-out I happened to pick".

## 12. The full forecast — 90 days ahead

In [ ]:
# Refit on ALL the data and forecast the next 90 days
final_model = ExponentialSmoothing(df["auto_rate"], trend="add",
                                    seasonal="add", seasonal_periods=7,
                                    initialization_method="estimated").fit()
future_idx = pd.date_range(df.index[-1] + pd.Timedelta(days=1), periods=90, freq="D")
forecast   = pd.Series(final_model.forecast(90).values, index=future_idx)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(df.index[-180:], df["auto_rate"].iloc[-180:], color="#4C72B0",
        lw=1.6, label="history (last 180d)")
ax.plot(forecast.index, forecast, color="#C44E52", lw=2.2, label="3-month forecast")
ax.axvline(df.index[-1], color="orange", lw=1, ls=":")
ax.set_title("Three-month forecast of the AI-bot's automation rate")
ax.set_ylabel("Automation rate")
ax.legend()
plt.tight_layout(); plt.show()

print(f"Last observed value : {df['auto_rate'].iloc[-1]:.3f}")
print(f"Forecast for day +30: {forecast.iloc[29]:.3f}")
print(f"Forecast for day +90: {forecast.iloc[89]:.3f}")


## 🧪 Practice exercises

### Exercise 1 — Monthly view

Resample the daily series to **monthly mean** and plot it. Then print the three highest-automation months across all 24 months.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
monthly = df["auto_rate"].resample("ME").mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly.index, monthly, marker="o", color="#4C72B0", lw=2)
ax.set_title("Monthly average automation rate")
ax.set_ylabel("Automation rate")
plt.tight_layout(); plt.show()

print("Top-3 months:")
print(monthly.sort_values(ascending=False).head(3).round(3))
```

The pattern `series.resample("ME").mean().sort_values(...)` is a one-liner for "rank periods by metric" — you'll use it for revenue, sign-ups, etc.
</details>

### Exercise 2 — 30-day rolling standard deviation

The **volatility** of a series is often as interesting as its level. Compute a 30-day rolling standard deviation of `auto_rate` and plot it. Where is the series most volatile?

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
vol = df["auto_rate"].rolling(30).std()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(vol.index, vol, color="#DD8452", lw=1.6)
ax.set_title("30-day rolling standard deviation (volatility)")
ax.set_ylabel("Standard deviation")
plt.tight_layout(); plt.show()

print(f"Most volatile date: {vol.idxmax().date()}  (std = {vol.max():.3f})")
```

The dates of high volatility tend to align with the *Christmas dips* we baked in — a small shock in the series shows up as elevated rolling-std weeks later.
</details>

### Exercise 3 — Beat the naive baseline

Build an even simpler forecast: **predict tomorrow as the average of the last 14 days**, day-by-day for the test set. Compute its MAE on the 30-day hold-out and compare with the existing baselines.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
# Predict each test day using the previous 14 days' mean, recursively
window = list(train.iloc[-14:])
preds  = []
for _ in range(h):
    preds.append(np.mean(window))
    window.pop(0); window.append(preds[-1])    # use our own prediction as the new "yesterday"

fc_ma14 = pd.Series(preds, index=test.index)
print(f"MA-14 recursive MAE: {(test - fc_ma14).abs().mean():.4f}")
```

This is a **recursive** forecast — the model's own predictions feed the next step. Recursive forecasts compound errors quickly, which is why a 14-day rolling mean drifts away from reality over a 30-day horizon. Direct multi-step forecasts (like Holt-Winters) usually do better.
</details>

### Exercise 4 — Debug me 🐞

The cell below tries to plot the daily series with weekday names on the x-axis, but the dates aren't being interpreted as dates. Find and fix the bug.

```python
bad = pd.DataFrame({"date": ["2024-01-01","2024-01-02","2024-01-03"],
                    "v":    [0.5, 0.6, 0.55]})
bad.set_index("date")["v"].plot()
```

In [ ]:
# Your fixed version  👇


<details>
<summary>💡 <b>Solution</b></summary>

The strings look like dates to *us* but pandas treats them as plain strings until you convert them.

```python
bad = pd.DataFrame({"date": ["2024-01-01","2024-01-02","2024-01-03"],
                    "v":    [0.5, 0.6, 0.55]})
bad["date"] = pd.to_datetime(bad["date"])    # <-- the missing step
bad = bad.set_index("date")
bad["v"].plot(marker="o")
```

**Rule of thumb.** The first thing you do after `pd.read_csv` is `df["date"] = pd.to_datetime(df["date"])`. Without that step, every time-aware pandas operation either fails or — worse — silently does the wrong thing.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise A — Anomaly detection on residuals

Decompose `df['auto_rate']` (you already did this in §7). Flag any *residual* values more than 3 standard deviations from zero as anomalies, and print their dates.


<details>
<summary>💡 <b>Solution</b></summary>

```python
resid = decomp.resid.dropna()
threshold = 3 * resid.std()
anomalies = resid[resid.abs() > threshold]
print(f"Flagged {len(anomalies)} anomalies (|resid| > {threshold:.3f}):")
print(anomalies.head(10).round(3))
```

**Why "anomaly on the residual" works.** The trend and seasonal
components explain the expected pattern; whatever the residual
catches is, by construction, *unexpected*. The Christmas dips
you baked in around day 357 of 2023/24 should show up clearly.

</details>

### Stretch exercise B — Forecast and compare two horizons

Forecast 14 days and 90 days ahead from the same Holt-Winters fit. Plot both forecasts on the same chart alongside the last 60 days of history. Comment on why long-horizon forecasts have wider uncertainty in practice.


<details>
<summary>💡 <b>Solution</b></summary>

```python
m = ExponentialSmoothing(df["auto_rate"], trend="add", seasonal="add",
                          seasonal_periods=7, initialization_method="estimated").fit()
f14 = m.forecast(14)
f90 = m.forecast(90)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(df.index[-60:], df["auto_rate"].iloc[-60:], color="#4C72B0", lw=1.6, label="last 60d")
ax.plot(f14.index, f14, color="#55A467", lw=2.5, label="14-day forecast")
ax.plot(f90.index, f90, color="#C44E52", lw=2.0, alpha=0.7, label="90-day forecast")
ax.set_ylabel("auto_rate"); ax.legend(); ax.set_title("Two forecast horizons")
plt.tight_layout(); plt.show()
```

**Why long-horizon uncertainty grows.** Errors compound at each
step — the model uses *its own predictions* as inputs to the
next step. By day 90 the prediction is built on 90 layers of
"approximately correct" inputs, so the variance grows roughly
with $\sqrt{h}$. Always report a *prediction interval*, not just
a point forecast, for anything beyond a few days.

</details>

## 🎁 Bonus mini-project — A weekly forecasting report

Write a function `weekly_forecast_report(daily_series, weeks=4)` that:

1. Resamples the daily series to weekly mean.
2. Fits a Holt-Winters model with weekly seasonality.
3. Forecasts `weeks` weeks ahead.
4. Returns a tidy DataFrame with columns `date`, `actual_or_forecast`, `kind` (history / forecast).
5. Plots the last 26 weeks of history and the forecast on one chart.

In [ ]:
# Your code here  👇


<details>
<summary>💡 <b>Solution</b></summary>

```python
def weekly_forecast_report(daily_series, weeks=4):
    weekly = daily_series.resample("W").mean()

    model = ExponentialSmoothing(weekly, trend="add", seasonal=None,
                                  initialization_method="estimated").fit()
    fc = model.forecast(weeks)

    history = weekly.rename("actual_or_forecast").to_frame()
    history["kind"] = "history"
    forecast = fc.rename("actual_or_forecast").to_frame()
    forecast["kind"] = "forecast"
    report = pd.concat([history, forecast])
    report.index.name = "date"

    last_26 = weekly.tail(26)
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.plot(last_26.index, last_26, color="#4C72B0", lw=2, label="history")
    ax.plot(fc.index,      fc,      color="#C44E52", lw=2, label=f"{weeks}-week forecast")
    ax.set_title(f"Weekly automation rate — {weeks}-week forecast")
    ax.set_ylabel("Automation rate")
    ax.legend()
    plt.tight_layout(); plt.show()

    return report


rep = weekly_forecast_report(df["auto_rate"], weeks=8)
print(rep.tail(10).round(3))
```

**What you built.** A function you can hand to a colleague: feed it any daily metric, get back a weekly forecast plus the chart. This is the smallest possible "production-shaped" forecasting tool — and it's 20 lines.
</details>

## 🧠 Key takeaways

1. **Set a `DatetimeIndex`** with `pd.to_datetime` + `set_index`. Everything else flows from there.
2. **Resample** (`.resample(rule).agg(...)`) to change granularity; **roll** (`.rolling(window)`) to smooth at fixed granularity.
3. **`.diff(k)`** and **`.shift(k)`** are your tools for change-over-time and lag features.
4. **Decompose** before modelling: trend + seasonality + residual exposes what your model needs to capture.
5. **Three baselines you must beat:** naive (last value), seasonal naive, moving average.
6. **Holt-Winters** handles trend + seasonality with three parameters — a great default.
7. **Walk-forward backtesting** is the honest evaluation. One hold-out is one lucky sample.
8. Forecasts are *probability distributions* hiding inside point estimates. Report confidence intervals when you can.

## ✅ Self-assessment

- [ ] Convert a date column to `datetime` and set it as the index
- [ ] Slice a DataFrame by a date string or date range
- [ ] Resample from daily to weekly with a custom aggregation
- [ ] Compute a 7-day and 30-day rolling mean
- [ ] Decompose a series into trend + seasonality + residual
- [ ] Build a naive, seasonal-naive, and Holt-Winters forecast on the same data
- [ ] Evaluate forecasts with MAE / RMSE / MAPE
- [ ] Run a walk-forward backtest

## 🚀 Next step

Continue with **Notebook 15 — Embeddings and Semantic Retrieval**, where you'll upgrade the keyword-matching RAG from NB11 to proper semantic search — and benchmark the two approaches on the same evaluation set.